In [1]:
import time

print("Starting test...")

start = time.time()
import pandas as pd
end = time.time()

print("Pandas imported:", pd.__version__)
print("Import took:", round(end - start, 2), "seconds")

Starting test...


Pandas imported: 3.0.6
Import took: 1.01 seconds


In [2]:
import pandas as pd

train_df = pd.read_parquet("../data/train.parquet")

print("Dataset shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

Dataset shape: (58516, 45)

Columns:
['appID', 'name', 'release_date', 'estimated_owners', 'peak_ccu', 'required_age', 'price', 'dlc_count', 'detailed_description', 'short_description', 'supported_languages', 'full_audio_languages', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'user_score', 'positive', 'negative', 'score_rank', 'achievements', 'recommendations', 'notes', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'screenshots', 'movies', 'packages', 'release_year', 'rank_in_year', 'games_in_year', 'hit']


In [3]:
# Check data types
print("DATA TYPES:")
print(train_df.dtypes)

# Check missing values in possible features
print("\nMISSING VALUES:")
print(train_df[
    ["price", "required_age", "windows", "mac", "linux",
     "dlc_count", "supported_languages", "achievements"]
].isna().sum())

# Check some example values
print("\nEXAMPLE VALUES:")
print(train_df[
    ["price", "required_age", "windows", "mac", "linux",
     "dlc_count", "supported_languages", "achievements"]
].head(10))

# Check target distribution
print("\nHIT DISTRIBUTION:")
print(train_df["hit"].value_counts())
print(train_df["hit"].value_counts(normalize=True) * 100)

DATA TYPES:
appID                                  str
name                                   str
release_date                datetime64[us]
estimated_owners                       str
peak_ccu                             int64
required_age                         int64
price                              float64
dlc_count                            int64
detailed_description                   str
short_description                      str
supported_languages                 object
full_audio_languages                object
reviews                                str
header_image                           str
website                                str
support_url                            str
support_email                          str
windows                               bool
mac                                   bool
linux                                 bool
metacritic_score                     int64
metacritic_url                         str
user_score                           int64

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

# Separate features and target
X_train = train_df[["price", "required_age", "windows", "mac", "linux"]]
y_train = train_df["hit"]

# Create and train the baseline model
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

# Predict on training data for now
dummy_predictions = dummy.predict(X_train)

print("Dummy Classifier")
print("Accuracy:", accuracy_score(y_train, dummy_predictions))
print("Hit recall:", recall_score(y_train, dummy_predictions, zero_division=0))

Dummy Classifier
Accuracy: 0.7517260236516509
Hit recall: 0.0


In [5]:
val_df = pd.read_parquet("../data/validation.parquet")

X_val = val_df[["price", "required_age", "windows", "mac", "linux"]]
y_val = val_df["hit"]

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score
import pandas as pd

features = ["price", "required_age", "windows", "mac", "linux"]

results = []

# Dummy baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_predictions = dummy.predict(X_val)

results.append({
    "Features": "Dummy baseline",
    "Accuracy": accuracy_score(y_val, dummy_predictions),
    "Hit Recall": recall_score(y_val, dummy_predictions, zero_division=0)
})

# Test one feature at a time
for feature in features:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    model.fit(X_train[[feature]], y_train)
    predictions = model.predict(X_val[[feature]])

    results.append({
        "Features": feature,
        "Accuracy": accuracy_score(y_val, predictions),
        "Hit Recall": recall_score(y_val, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)

print(results_df.round(3))

         Features  Accuracy  Hit Recall
0  Dummy baseline     0.748       0.000
1           price     0.746       0.000
2    required_age     0.751       0.027
3         windows     0.748       0.000
4             mac     0.748       0.000
5           linux     0.748       0.000


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

# Create model using all five features
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

# Train model
model.fit(X_train, y_train)

# Predict validation data
predictions = model.predict(X_val)

print("Logistic Regression - All Features")
print("Accuracy:", accuracy_score(y_val, predictions))
print("Hit Recall:", recall_score(y_val, predictions))

Logistic Regression - All Features
Accuracy: 0.6832256741515431
Hit Recall: 0.3276283618581907


In [8]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    f1_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_val, predictions))

print("\nAdditional Metrics:")
print("Precision:", precision_score(y_val, predictions))
print("F1-score:", f1_score(y_val, predictions))

Confusion Matrix:
[[11719  2879]
 [ 3300  1608]]

Additional Metrics:
Precision: 0.35836862045910406
F1-score: 0.34230973922299096


In [9]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get predicted probabilities
probabilities = model.predict_proba(X_val)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:
    threshold_predictions = (probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, threshold_predictions),
        "Precision": precision_score(y_val, threshold_predictions, zero_division=0),
        "Recall": recall_score(y_val, threshold_predictions, zero_division=0),
        "F1-score": f1_score(y_val, threshold_predictions, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(3))

   Threshold  Accuracy  Precision  Recall  F1-score
0        0.3     0.252      0.252   1.000     0.402
1        0.4     0.252      0.252   1.000     0.402
2        0.5     0.683      0.358   0.328     0.342
3        0.6     0.710      0.355   0.188     0.246
4        0.7     0.748      0.472   0.028     0.052


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

feature_groups = {
    "Price + Age": ["price", "required_age"],
    "Platform support": ["windows", "mac", "linux"],
    "Price + Platforms": ["price", "windows", "mac", "linux"],
    "All five features": ["price", "required_age", "windows", "mac", "linux"]
}

combination_results = []

for name, selected_features in feature_groups.items():

    model_test = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )

    model_test.fit(X_train[selected_features], y_train)

    predictions_test = model_test.predict(X_val[selected_features])

    combination_results.append({
        "Feature group": name,
        "Accuracy": accuracy_score(y_val, predictions_test),
        "Precision": precision_score(y_val, predictions_test, zero_division=0),
        "Recall": recall_score(y_val, predictions_test, zero_division=0),
        "F1-score": f1_score(y_val, predictions_test, zero_division=0)
    })

combination_df = pd.DataFrame(combination_results)

print(combination_df.round(3))

       Feature group  Accuracy  Precision  Recall  F1-score
0        Price + Age     0.699      0.381   0.311     0.342
1   Platform support     0.687      0.333   0.243     0.281
2  Price + Platforms     0.677      0.356   0.351     0.353
3  All five features     0.683      0.358   0.328     0.342


In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

best_features = ["price", "windows", "mac", "linux"]

best_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
)

best_model.fit(X_train[best_features], y_train)

best_predictions = best_model.predict(X_val[best_features])

print("Selected Model: Price + Platforms")
print("Accuracy:", accuracy_score(y_val, best_predictions))
print("Precision:", precision_score(y_val, best_predictions))
print("Recall:", recall_score(y_val, best_predictions))
print("F1-score:", f1_score(y_val, best_predictions))

Selected Model: Price + Platforms
Accuracy: 0.6770737209063877
Precision: 0.35609352369128905
Recall: 0.3506519967400163
F1-score: 0.3533518119289601


In [12]:
test_df = pd.read_parquet("../data/test.parquet")

In [13]:
print("Selected features:", best_features)
print("Training games:", len(X_train))
print("Validation games:", len(X_val))
print("Test games:", len(test_df))

Selected features: ['price', 'windows', 'mac', 'linux']
Training games: 58516
Validation games: 19506
Test games: 19506


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

X_test = test_df[best_features]
y_test = test_df["hit"]

test_predictions = best_model.predict(X_test)

print("Final test results")
print("------------------")
print("Accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("Precision:", round(precision_score(y_test, test_predictions, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, test_predictions, zero_division=0), 3))
print("F1-score:", round(f1_score(y_test, test_predictions, zero_division=0), 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))

Final test results
------------------
Accuracy: 0.678
Precision: 0.366
Recall: 0.363
F1-score: 0.364

Confusion matrix:
[[11434  3118]
 [ 3156  1798]]
